#15 — Gold Performance Optimization: Liquid Clustering vs Partition+Z-Order

## Configuration

In [0]:
# Benchmarks fact_street_readings (the largest table, ~87.8M source rows —
# the one where clustering strategy actually matters) two ways:
#   A) CLUSTER BY (street_id, reading_date) — Liquid Clustering
#   B) PARTITIONED BY reading_date-derived year_month, ZORDER BY street_id
# 5 query patterns x 3 runs each, results saved to benchmark_results.

import time
from pyspark.sql import functions as F

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("gold_schema", "gold", "2. Gold Schema")
CATALOG = dbutils.widgets.get("catalog_name")
GOLD = dbutils.widgets.get("gold_schema")

FACT = f"{CATALOG}.{GOLD}.fact_street_readings"
FACT_LIQUID = f"{CATALOG}.{GOLD}.fact_street_readings_liquid"
FACT_PART = f"{CATALOG}.{GOLD}.fact_street_readings_partitioned"
BENCH_TABLE = f"{CATALOG}.{GOLD}.benchmark_results"
DIM_STREET = f"{CATALOG}.{GOLD}.dim_street"

source_rows = spark.table(FACT).count()
print(f"Source: {FACT} ({source_rows:,} rows)")

## Approach A: Liquid Clustering

In [0]:
# Cluster keys: street_id (INT FK, the most selective/common filter) +
# reading_date (used in range scans). No listing_year-equivalent needed —
# reading_date itself is low-enough cardinality (284 distinct days) to
# cluster on directly.

spark.sql(f"""
CREATE OR REPLACE TABLE {FACT_LIQUID}
USING DELTA
CLUSTER BY (street_id, reading_date)
TBLPROPERTIES (
  'quality' = 'gold',
  'optimization' = 'liquid_clustering',
  'cluster_keys' = 'street_id_reading_date',
  'delta.enableChangeDataFeed' = 'true'
)
AS SELECT * FROM {FACT}
""")
spark.sql(f"OPTIMIZE {FACT_LIQUID}")
print(f"Liquid table ready: {spark.table(FACT_LIQUID).count():,} rows")

## Approach B: Partitioning + Z-Order

In [0]:
# Partition by year_month (low-cardinality, ~10 partitions across the real
# data window — reading_date itself, 284 distinct values, would be far too
# high-cardinality to partition on directly). Z-order by street_id.

spark.sql(f"""
CREATE OR REPLACE TABLE {FACT_PART}
USING DELTA
PARTITIONED BY (year_month)
TBLPROPERTIES (
  'quality' = 'gold',
  'optimization' = 'partition_zorder',
  'partition_key' = 'year_month',
  'zorder_keys' = 'street_id',
  'delta.enableChangeDataFeed' = 'true'
)
AS SELECT *, date_format(reading_date, 'yyyy-MM') AS year_month FROM {FACT}
""")
spark.sql(f"OPTIMIZE {FACT_PART} ZORDER BY (street_id)")
print(f"Partitioned table ready: {spark.table(FACT_PART).count():,} rows")
spark.table(FACT_PART).groupBy("year_month").count().orderBy("year_month").show(20, truncate=False)


## Benchmarking

In [0]:
def benchmark(query, label, runs=3):
    times = []
    for i in range(runs):
        start = time.perf_counter()
        spark.sql(query).collect()
        elapsed = round(time.perf_counter() - start, 3)
        times.append(elapsed)
        print(f"    Run {i+1}: {elapsed}s")
    avg = round(sum(times) / len(times), 3)
    print(f"  [{label}] Average: {avg}s")
    return avg


QUERIES = [
    ("Q1: Single street_id filter",
     f"SELECT * FROM {FACT_LIQUID} WHERE street_id = 1",
     f"SELECT * FROM {FACT_PART} WHERE street_id = 1"),

    ("Q2: Multi-street_id + date range",
     f"SELECT street_id, AVG(pollution) FROM {FACT_LIQUID} "
     f"WHERE street_id IN (1,5,10) AND reading_date BETWEEN '2023-07-01' AND '2023-07-31' "
     f"GROUP BY street_id",
     f"SELECT street_id, AVG(pollution) FROM {FACT_PART} "
     f"WHERE street_id IN (1,5,10) AND reading_date BETWEEN '2023-07-01' AND '2023-07-31' "
     f"GROUP BY street_id"),

    ("Q3: Single-day scan across all streets (partition home turf)",
     f"SELECT street_id, COUNT(*), AVG(noise) FROM {FACT_LIQUID} "
     f"WHERE reading_date = '2023-08-15' GROUP BY street_id",
     f"SELECT street_id, COUNT(*), AVG(noise) FROM {FACT_PART} "
     f"WHERE reading_date = '2023-08-15' GROUP BY street_id"),

    ("Q4: Full aggregation by street (scan-heavy, dim_street join)",
     f"SELECT d.street_name, COUNT(*) AS readings, AVG(f.pollution) AS avg_pollution "
     f"FROM {FACT_LIQUID} f JOIN {DIM_STREET} d ON f.street_id = d.street_id AND d.__END_AT IS NULL "
     f"GROUP BY d.street_name ORDER BY readings DESC",
     f"SELECT d.street_name, COUNT(*) AS readings, AVG(f.pollution) AS avg_pollution "
     f"FROM {FACT_PART} f JOIN {DIM_STREET} d ON f.street_id = d.street_id AND d.__END_AT IS NULL "
     f"GROUP BY d.street_name ORDER BY readings DESC"),

    ("Q5: Rain event lookup (raining_clipped > 50)",
     f"SELECT street_id, reading_date, raining_clipped FROM {FACT_LIQUID} WHERE raining_clipped > 50",
     f"SELECT street_id, reading_date, raining_clipped FROM {FACT_PART} WHERE raining_clipped > 50"),
]

benchmark_results = []
for label, q_liq, q_part in QUERIES:
    print(f"\n{label}")
    print("  Liquid Clustering:")
    t_liq = benchmark(q_liq, "Liquid")
    print("  Partitioned + Z-Order:")
    t_part = benchmark(q_part, "Z-Order")
    winner = "Liquid" if t_liq <= t_part else "Z-Order"
    speedup = round(max(t_liq, t_part) / min(t_liq, t_part), 2) if min(t_liq, t_part) > 0 else 1.0
    print(f"  Winner: {winner} ({speedup}x)")
    benchmark_results.append((label, float(t_liq), float(t_part), winner, speedup))

schema = "query STRING, liquid_avg_secs DOUBLE, zorder_avg_secs DOUBLE, winner STRING, speedup DOUBLE"
(spark.createDataFrame(benchmark_results, schema)
 .withColumn("benchmarked_at", F.current_timestamp())
 .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(BENCH_TABLE))
spark.sql(f"COMMENT ON TABLE {BENCH_TABLE} IS 'Benchmark: Liquid Clustering vs Partition+Z-Order on fact_street_readings, 5 queries x 3 runs'")

liquid_wins = sum(1 for r in benchmark_results if r[3] == "Liquid")
print(f"\n{'='*60}")
print(f"  Liquid wins : {liquid_wins}/{len(benchmark_results)}")
print(f"  Z-Order wins: {len(benchmark_results) - liquid_wins}/{len(benchmark_results)}")
print(f"{'='*60}")
print("  Use Liquid Clustering when filters span multiple keys unpredictably")
print("  (street_id + date range together, ad-hoc BI queries).")
print("  Use Partition+Z-Order when one column is near-always in WHERE")
print("  (single-day lookups — Q3 is partitioning's home turf).")
display(spark.table(BENCH_TABLE).orderBy("query"))
